# W07A Colab B · 내용·평가·마지막 웹 앱

딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2 · 2026년10월13일

각 노트북은 독립 실행됩니다. CPU에서 위부터 실행하세요. GPU·유료 API·파일 업로드가 필요 없습니다. 파일 → Drive에 사본 저장으로 자신의 실습본을 만드세요. 설치 후 재시작 안내가 나오면 재시작하고 첫 셀부터 다시 실행합니다.

설명용 데이터와 연습문제 데이터는 별개입니다. 문제 코드는 읽고 수정할 대상으로 markdown에 제시합니다. 자신의 답 문자열이나 코드가 비어 있어도 자동 시연과 검증은 독립 실행됩니다. 전체 실행 성공은 자신의 풀이가 정답이라는 뜻이 아닙니다.


## 코드 정의와 입력·출력

외부에서 가져오는 함수의 이름을 아래에서 클릭하면 설치한 고정 버전의 실제 정의 행을 볼 수 있습니다. 함수의 docstring을 먼저 읽고 입력과 반환값을 대조하세요. 함수 호출은 편의를 위한 것이며 Q40–Q60은 기본 Python만으로 풉니다.

| 함수 | 입력 | 출력·의미 |
|---|---|---|
| review_data | 없음 | train9행/test3행, 집단·장르를 담은 새 dict; 합성 표 자동 생성 |
| rating_stats | (사용자,영화,평점) 리스트, 선택 groups dict | 영화 또는 (집단,영화)별 sum/count/mean; 중복·잘못된 평점 거부 |
| mean_predict | train, 영화ID, 선택 집단, min_count | (평점,근거); 집단→영화→학습전체; test는 입력하지 않음 |
| cosine_score | 같은 길이의 두 수치 리스트 | 코사인 또는 영벡터일 때 None; 특징 순서 일치 필요 |
| rank_candidates | data dict, method/group/min_count/top_k | 미관람 후보를 조건 적용 후 정렬한 dict 목록 |
| rating_metrics | 같은 순서의 실제·예측 평점 리스트 | MAE/RMSE dict; 평가 관측 수로 나눔 |
| ranking_metrics | 추천ID·관련집합·카탈로그, k | P/R/NDCG/coverage; 짧은 목록의 P 분모도 요청K |
| review_experiment | 방법·최소수·K·집단 | 고정 합성 실험 결과 dict, 예측표·목록·지표 |
| review_view | 방법·최소수·K·집단, 선택 report_fn | 설명·추천표·예측표·지표표·그림; 실제 앱 callback |
| build_review_app | 선택 report_fn(result)->문자열 | Gradio 앱; 생성과 launch는 별도 |
| definition_link | 수업 함수/클래스/메소드, 고정ref | GitHub 정의 시작 행 URL; 파일·네트워크 변경 없음 |

이 함수들은 숨은 학습이나 파일 다운로드를 하지 않습니다. 핵심 데이터는 코드에 내장된 합성 표이며 입력과 출력에 명시된 통계만 계산합니다. 빈 학습 목록은 평균을 꾸미지 않고 오류로 보고합니다. 집단 목록 전체 대체와 쌍별 평점 대체의 적용 단위를 구별하세요.


In [ ]:
import os,sys,subprocess,math
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w07a'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.26.0'],check=True)
from IPython.display import display,Markdown
from luna_recsys.review_foundations import (review_data,rating_stats,mean_predict,
    cosine_score,rank_candidates,rating_metrics,ranking_metrics,review_experiment)
from luna_recsys.review_app import review_view,build_review_app
from luna_recsys.model_comparison import definition_link
print('Python',sys.version.split()[0], '수업 버전',COURSE_VERSION)


In [ ]:
apis=[review_data,rating_stats,mean_predict,cosine_score,rank_candidates,
      rating_metrics,ranking_metrics,review_experiment,review_view,build_review_app,definition_link]
display(Markdown('\n'.join(f'- [{f.__name__}]({definition_link(f,COURSE_VERSION)})' for f in apis)))


## 1. 원자료를 먼저 읽기

아래 합성 표는 처음 실행할 때 자동 생성됩니다. 각 행은 (사용자,영화,평점)이며 train은 통계, test는 평가 전용입니다. 사용자u1/u2는G1, u3/u4는G2입니다. 영화 특징 순서는 [액션,코미디]입니다. 데이터를 눈으로 읽은 뒤 계산하세요.


In [ ]:
demo=review_data()
print('train',demo['train'])
print('test',demo['test'])
print('groups',demo['groups'])
print('features',demo['features'])


## 2. 방향을 비교하고 같은 평가 조건으로 보기

u1이 학습에서 좋아한 영화(4점 이상)는A이며 그 특징은[1,0]입니다. 프로필과 후보의 코사인을 비교합니다. 코사인 점수는 예상 별점이 아니므로 실제 별점과 빼서 RMSE를 계산하지 않습니다.


In [ ]:
demo_profile=[1.,0.]
for item in ['B','D','E']:
    print(item,cosine_score(demo_profile,demo['features'][item]))
for method in ['mean','count','group','content']:
    result=review_experiment(method,1,3,'G1')
    print(method,result['ranked'],result['metrics'])


## 3. SVD 마무리 재확인

지난 W06B의 재구성 함수를 같은 버전에서 가져옵니다. `teaching_matrix("main")`은2×2 새 배열, `svd_steps(A,rank)`는U/s/Vt·근사·오차·에너지 비율을 담은dict를 반환합니다. 파일·학습 상태는 없습니다. 고유값이나 특이값의 부호·차원은 보충노트와 연결해서 읽으세요.


In [ ]:
from luna_recsys.svd_foundations import teaching_matrix,svd_steps
from luna_recsys.svd_lab import reconstruction_view
for fn in [teaching_matrix,svd_steps,reconstruction_view]:
    display(Markdown(f'[{fn.__name__}]({definition_link(fn,COURSE_VERSION)})'))
import numpy as np
svd_a=teaching_matrix('main')
for rank in [1,2]:
    svd_result=svd_steps(svd_a,rank)
    print('rank',rank,svd_result)
U,s,Vt=np.linalg.svd(svd_a,full_matrices=False)
assert np.allclose(U@np.diag(s)@Vt,svd_a)


### S1 · 재구성 오차 해석

k=1→2에서 원소와 오차가 어떻게 바뀌나요? 그 변화가 미관측 평점 예측 개선을 증명하는지 설명하세요. 힌트: 입력 행렬과 별도 test 정답의 역할을 구별합니다. 자가 점검: 재구성 오차와 예측 오차를 각각 무엇과 비교했는지 적으세요.


In [ ]:
svd_reason=''


## 4. 중간고사 대비 Python 연습

문제 번호는 학생 Notion·PDF와 같습니다. 먼저 출력이나 수정 방향을 종이에 적고 자신의 코드 셀에서 확인하세요. 문제의 잘못된 코드는 자동으로 실행하지 않습니다. 문법 설명은 각 문제에 제공되어 있습니다.


### Q54 · 내적 빈칸 (빈칸 / 적용)

같은 위치의 특징끼리 곱해 더하도록 ___을 채워라. range(2)는0,1을 차례로 준다.

```python
x=[1,2]
y=[3,1]
dot=0
for j in range(2):
    dot += ___
print(dot)
```

목표: 내용·벡터·TF–IDF의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q54_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q55 · 코사인 분모 오류 (오류 / 적용)

**0.5는 제곱근이다. 아래 코드는 두 벡터 길이의 합으로 나눈다. 두 길이의 곱으로 수정하고 소수 넷째 자리까지 출력하라.

```python
x=[1,1]
y=[1,0]
dot=1
nx=(1**2+1**2)**0.5
ny=(1**2+0**2)**0.5
sim=dot/(nx+ny)
print(round(sim,4))
```

목표: 내용·벡터·TF–IDF의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q55_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q56 · 프로필 출력 추적 (출력 / 적용)

두 선호 영화의 특징을 같은 가중치로 평균한다. 코드를 실행하지 않고 출력 리스트와 각 좌표의 의미를 쓰라.

```python
liked=[[1,0],[1,1]]
profile=[0,0]
for v in liked:
    for j in range(2):
        profile[j]+=v[j]
for j in range(2):
    profile[j]/=len(liked)
print(profile)
```

목표: 내용·벡터·TF–IDF의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q56_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q57 · DF 출력 추적 (출력 / 적용)

`set(words)`는 한 문서의 중복 단어를 제거한다. df의 최종값과 중복 제거 이유를 쓰라.

```python
docs=[['robot','robot'],['robot','space'],['space']]
df=0
for words in docs:
    if 'robot' in set(words):
        df+=1
print(df)
```

목표: 내용·벡터·TF–IDF의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q57_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q58 · MAE 빈칸 (빈칸 / 적용)

abs는 절댓값이다. ___을 채워 MAE를 계산하라. y와 p는 같은 관측 순서이다.

```python
y=[5,2,4]
p=[4,4,4]
errors=[]
for j in range(len(y)):
    errors.append(abs(y[j]-p[j]))
mae=sum(errors)/___
print(mae)
```

목표: 분할·평가의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q58_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q59 · 학습 평균의 누수 오류 (오류 / 적용)

test는 평가 정답이다. 누수 없이 예측하도록 잘못된 줄을 고치고 예측값과 MAE를 쓰라.

```python
train=[1,3]
test=[5]
pred=sum(train+test)/len(train+test)
mae=sum(abs(y-pred) for y in test)/len(test)
print(pred,mae)
```

목표: 분할·평가의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q59_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


### Q60 · 정확도와 커버리지 출력 추적 (출력 / 통합)

관련 집합은 {B,C,D,E}, 추천 목록은 [A,B,C], 전체 카탈로그는 [A,B,C,D,E,F]이다. 코드를 실행하지 않고 p,r,coverage를 구하라. coverage는 이 한 사용자 목록의 카탈로그 커버리지이다.

```python
recommended=['A','B','C']
relevant=['B','C','D','E']
catalog=['A','B','C','D','E','F']
hits=0
for item in recommended:
    if item in relevant:
        hits+=1
p=hits/len(recommended)
r=hits/len(relevant)
coverage=len(set(recommended))/len(catalog)
print(round(p,4),r,coverage)
```

목표: 분할·평가의 계산 규칙과 코드를 연결합니다. 힌트: 각 변수의 현재 값과 조건의 참·거짓을 한 줄씩 적으세요. 자가 점검: 문제에 주어진 분모·후보 조건·입력 순서를 모두 사용했는지 확인하세요.


In [ ]:
q60_my_work=''  # 이 셀에 자신의 계산·설명을 기록하고 별도 코드 셀에서 풀이를 확인하세요.


## 5. 실행 결과 검증

아래 검사는 실습의 핵심 기능과 평가 경계를 확인합니다. 자신의 연습문항 정답을 자동 채점하는 셀은 아닙니다. 오류가 나면 출력 내용을 확인하고 중단 원인을 해결하세요. 실패를 빈 결과로 바꾸지 않습니다.


In [ ]:
check_data=review_data()
assert not {(u,i) for u,i,_ in check_data['train']}&{(u,i) for u,i,_ in check_data['test']}
assert rating_stats(check_data['train'])['B']['mean']==2.5
assert mean_predict(check_data['train'],'B',groups=check_data['groups'],group='G1',min_count=2)==(2.5,'item mean')
assert [r['item'] for r in rank_candidates(check_data,min_count=2)]==['E','B']
assert cosine_score([0,0],[1,0]) is None
check_result=review_experiment('content',2,2,'G1')
assert 'RMSE' not in check_result['metrics']
assert len(check_result['ranked'])==2
import matplotlib.pyplot as plt
check_view=review_view('group',2,2,'G1',lambda r:'callback verified')
assert 'callback verified' in check_view[0] and check_view[1][0][0]=='E'
plt.close(check_view[-1])
for rank in [1,2]:
    check_svd=reconstruction_view('main',rank)
    assert len(check_svd)==3
    plt.close(check_svd[-1])
print('W07A-B RUNTIME CHECK PASSED: train-only statistics, candidates, fallback, metrics.')


## 점검 퀴즈와 가져갈 생각

1. Q54를 다시 설명하세요: 내적 빈칸에서 핵심 조건은 무엇인가요?
2. Q55를 다시 설명하세요: 코사인 분모 오류에서 핵심 조건은 무엇인가요?
3. Q56를 다시 설명하세요: 프로필 출력 추적에서 핵심 조건은 무엇인가요?
4. Q57를 다시 설명하세요: DF 출력 추적에서 핵심 조건은 무엇인가요?
5. Q58를 다시 설명하세요: MAE 빈칸에서 핵심 조건은 무엇인가요?

위 점검 질문은 이미 푼 동일 번호 문항을 다시 말로 설명하는 활동입니다. 새 문항이 아닙니다. 한 행의 의미, 관측 분모, 후보 처리 순서, 평가 정답의 분리를 기억하세요.

참고: 주교재1–2장, 기존W01–W02 수업. [Google 내용 기반](https://developers.google.com/machine-learning/recommendation/content-based/basics), [scikit-learn 누수](https://scikit-learn.org/stable/common_pitfalls.html), [NumPy SVD](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html).


## 마지막: 직접 바꾸는 웹 애플리케이션

**LAB-B** · 최소 평점 수를1→2로 바꾸기 전에 빠질 후보를 예상하세요. 앱의 train 표와 추천 근거를 대조한 다음 아래 `my_report`에 설정과 반환 개수를 설명하는 문장을 추가하세요. 힌트: `result["min_count"]`와 `len(result["ranked"])`를 읽습니다. 자가 점검: 설정을 다시 바꾸면 설명도 실제 값에 맞게 바뀌어야 합니다. 코드 수정 후 앱 생성과 실행 셀을 다시 실행하세요.


In [ ]:
def my_report(result):
    return '관측 수와 후보 제외 이유를 표에서 확인하세요.'
app=build_review_app(report_fn=my_report)


Colab에서는 아래 셀 실행 뒤 내장 앱과 임시 공유 링크가 표시됩니다. 공유 링크는 만료될 수 있으므로 다음 실행은 이 고정 노트북 주소에서 시작하세요. 처음에는 mean으로 실행하고 다음에 count/group/content와 SVD 탭을 비교하세요. 앱을 닫아도 앞의 계산 결과는 남습니다.


In [ ]:
if IN_COLAB:
    app.launch(share=True,debug=False,prevent_thread_lock=True)
else:
    print('Local verification: app built. Browser checks are recorded separately.')
